# 01 DEMs

ICESat-2 ATL06 download and outlier removal, coregistration of the SRTM 1999 and IfSAR 2002/2012 DEMs, and ICESat-2-derived DEMs for 2019 and 2022.

Writes `../External_Data/ICESat-2/icesat_2_data.shp`, `../Data_Paper/ICESat-2/icesat_2_data_no_outliers.shp`, `../Data_Paper/DEMs/d1999_coregistered.tif`, `d2002_coregistered.tif`, `d2012_coregistered.tif`, `d2019_icesat2.tif`, `d2022_icesat2.tif`.

In [ ]:
import logging
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
import geoutils as gu
import xdem
from scipy.interpolate import RBFInterpolator, interp1d
from sliderule import sliderule, icesat2
from tqdm import tqdm

# Basic configuration to simply print info
logging.basicConfig(level=logging.INFO)

## 1) Download ICESat-2 ATL06 (SlideRule)

ATL06 points over the 2002 DEM footprint, keeping `atl06_quality_summary == 0`. Writes `../External_Data/ICESat-2/icesat_2_data.shp`.

In [ ]:
# Open example DEM
dem = xdem.DEM("../External_Data/raw_DEMs/2002-08-04.tif")

# Define region of interest as DEM footprint
bounds = list(dem.get_bounds_projected(4326))
region = sliderule.toregion(bounds)["poly"]

# Initialize SlideRule client
sliderule.init("slideruleearth.io")

# Define desired parameters for ICESat-2 ATL06
params = {
    "poly": region,
    "srt": icesat2.SRT_LAND,  # Surface-type
    "cnf": icesat2.CNF_SURFACE_HIGH,  # Confidence level
    "ats": 20.0,  # Minimum along-track spread
    "cnt": 10,  # Minimum count
}

# Request ATL06 subsetting in parallel
gdf = icesat2.atl06sp(params)

gdf = gdf[np.isfinite(gdf["h_li"])] # Keep valid data
gdf = gdf[gdf["atl06_quality_summary"]==0]  # Keep very high-confidence data
gdf = gdf.to_crs(dem.crs)
gdf.to_file("../External_Data/ICESat-2/icesat_2_data.shp")

## 2) Load DEMs and stable-terrain mask

2002 DEM (ellipsoid), 2012 DEM converted from NAVD88 to the ellipsoid with GEOID09. Stable terrain: off-glacier (RGI 6.0), slope < 20 degrees, 200-1500 m.

In [ ]:
dem = xdem.DEM("../External_Data/raw_DEMs/2002-08-04.tif")

d2012 = xdem.DEM('../External_Data/raw_DEMs/2012-08-30.tif').reproject(dem)
geoid_09 = gu.Raster("../External_Data/raw_DEMs/us_noaa_geoid09_ak.tif").reproject(d2012)
d2012 = d2012 + geoid_09


# Derive inlier mask as glaciers
glacier_outlines = gu.Vector('../External_Data/RGI60/01_rgi60_Alaska.shp')
inlier_mask = ~glacier_outlines.create_mask(dem)


# Remove terrain where slope > 20 degrees, where dem is below 100m (remove trees) and above 1500m (remove unstable slopes), and where 2012-2002 is too big (glaciers unacounted for)
slope = dem.slope()
slope_mask = slope.data < 20
# Combine the inlier mask and slope mask
inlier_mask.data = inlier_mask.data & slope_mask & (dem.data > 200) & (dem.data < 1500)
# Remove invalid entries from the inlier mask
inlier_mask.data[dem.data.mask==True]=False
inlier_mask.data[d2012.data.mask==True]=False

In [ ]:
gdf = gpd.read_file('../External_Data/ICESat-2/icesat_2_data.shp').to_crs(dem.crs)
gdf['datetime'] = pd.to_datetime(gdf['time'], format='%Y-%m-%dT%H:%M:%S.%fZ')

## 3) Remove ICESat-2 outliers

First RBF DEM from all points (multiquadric, epsilon = 100, smoothing = 500, 1 % random subsample); points whose difference from it exceeds the 95th percentile of their 50 m elevation bin are discarded. Writes `../Data_Paper/ICESat-2/icesat_2_data_no_outliers.shp`.

In [ ]:
def interpolate_icesat2(var, pts_gdf, dem_ref, factor, gridsize, function, epsilon=None, sm = 0):

    """
    Interpolate DataFrame data onto a regular grid using RBF interpolation.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame with columns 'X', 'Y', 'h_li'
    gridsize : int
        Size of the output grid (gridsize x gridsize)
    function : str
        RBF function type ('thin_plate_spline', 'multiquadric', 'inverse_multiquadric', 
                          'inverse_quadratic', 'gaussian', 'linear', 'cubic', 'quintic')
    
    Returns:
    --------
    X_grid, Y_grid : numpy.ndarray
        Grid coordinates
    Z_grid : numpy.ndarray
        Interpolated values on the grid
    """

    pts_gdf['X'] = [pt.x for pt in pts_gdf.geometry]
    pts_gdf['Y'] = [pt.y for pt in pts_gdf.geometry]


    N = len(pts_gdf)
    n_keep = N // factor

    random_indices = np.random.default_rng(42).choice(N, size=n_keep, replace=False)  # fixed seed

    
    # Extract coordinates and values
    points = pts_gdf[['X', 'Y']].values[random_indices]
    values = pts_gdf[var].values[random_indices]
    # Ensure there are no NaNs
    mask = ~np.isnan(values)
    points = points[mask]
    values = values[mask]
    
    # Create regular grid
    x_min, x_max = pts_gdf['X'].min(), pts_gdf['X'].max()
    y_min, y_max = pts_gdf['Y'].min(), pts_gdf['Y'].max()
    
    x_grid = np.linspace(x_min, x_max, gridsize)
    y_grid = np.linspace(y_min, y_max, gridsize)
    X_grid, Y_grid = np.meshgrid(x_grid, y_grid)
    
    # Flatten grid for interpolation
    grid_points = np.column_stack([X_grid.ravel(), Y_grid.ravel()])
    
    # Create RBF interpolator
    rbf = RBFInterpolator(points, values, kernel=function, epsilon=epsilon, smoothing=sm)
    
    # Interpolate onto grid
    Z_flat = rbf(grid_points)
    Z_grid = Z_flat.reshape(gridsize, gridsize)
    
    # Create transform from gridx and gridy
    transform = rio.transform.from_bounds(np.min(X_grid), np.min(Y_grid), np.max(X_grid), np.max(Y_grid), Z_grid.shape[1], Z_grid.shape[0])
    rast_rbf = xdem.DEM.from_array(data=np.flip(Z_grid, axis = 0), transform=transform, crs=pts_gdf.crs, nodata=np.nan)

    rast_rbf = rast_rbf.reproject(res=dem_ref.res, bounds=dem_ref.bounds)
    rast_rbf.set_vcrs('Ellipsoid')


    return rast_rbf

In [ ]:
rast_icesat = interpolate_icesat2('h_li', gdf, dem, 100, 1000, 'multiquadric', epsilon = 100, sm = 500)

In [ ]:
gdf['X'] = [pt.x for pt in gdf.geometry]
gdf['Y'] = [pt.y for pt in gdf.geometry]

# Interpolate the DEM at the points
vals = rast_icesat.interp_points(points=(gdf['X'], gdf['Y']))

In [ ]:
# Calculate the differences
diffs = np.abs(gdf['h_li'] - vals)

# Define bins (0–5000 every 50)
bins = np.arange(0, 5000 + 50, 50)

# Assign each point to a bin
bin_ids = np.digitize(gdf['h_li'].values, bins)

# Compute 95th percentile for each bin
bin_p95 = {}
for b in np.unique(bin_ids):
    mask = bin_ids == b
    if np.any(mask):
        bin_p95[b] = np.nanpercentile(diffs[mask], 95)

# Create threshold array matching each point
thresholds = np.array([bin_p95.get(b, np.nan) for b in bin_ids])

# Final mask
outlier_mask = diffs < thresholds

# Create geodataframe without outliers
gdf_no_outliers = gdf[outlier_mask].copy()
gdf_no_outliers.to_file('../Data_Paper/ICESat-2/icesat_2_data_no_outliers.shp')

## 4) Coregister DEMs: first to ICESat-2, then to the 2002 DEM

Nuth and Kääb: SRTM 1999 and IfSAR 2002 to the filtered ICESat-2 points; IfSAR 2012 to the coregistered 2002 DEM. Second SRTM pass onto the coregistered 2002 DEM (vertical shift + Nuth and Kääb + 2nd-order deramp, stable terrain 1-1000 m). Writes `../Data_Paper/DEMs/d1999_coregistered.tif`, `d2002_coregistered.tif`, `d2012_coregistered.tif`.

In [ ]:
gdf_no_outliers = gpd.read_file('../Data_Paper/ICESat-2/icesat_2_data_no_outliers.shp')


d1999 = xdem.DEM('../External_Data/raw_DEMs/1999.tif').reproject(d2012) # Already in Ellipsoid
d1999.set_vcrs('Ellipsoid')

d2002 = dem.copy()
d2002.set_vcrs('Ellipsoid')
d2012.set_vcrs('Ellipsoid')


# Initialize coregistration pipeline
coreg_pipeline = xdem.coreg.NuthKaab()

# Fit and apply the coregistration
d1999_coregistered = coreg_pipeline.fit_and_apply(reference_elev=gdf_no_outliers, to_be_aligned_elev=d1999, inlier_mask=inlier_mask, z_name="h_li", random_state=42)
d1999_coregistered.set_vcrs('Ellipsoid')
d2002_coregistered = coreg_pipeline.fit_and_apply(reference_elev=gdf_no_outliers, to_be_aligned_elev=d2002, inlier_mask=inlier_mask, z_name="h_li", random_state=42)
d2002_coregistered.set_vcrs('Ellipsoid')


# 2nd coregistration with d2002
# We have to modify the inlier_mask to exclude NaNs
inlier_mask.data[d2012.data.mask] = False
d2012_coregistered = coreg_pipeline.fit_and_apply(reference_elev=d2002_coregistered, to_be_aligned_elev=d2012, inlier_mask=inlier_mask, random_state=42) # Coregister to d2002 because somehow icesat fails
d2012_coregistered.set_nodata(np.nan)
d2012_coregistered.set_vcrs('Ellipsoid')

In [ ]:
# 2nd Iteration of coregistration because d1999 has massive bias at low elevations

# Remove terrain where slope > 20 degrees, where dem is below 100m (remove trees) and above 1500m (remove unstable slopes), and where 2012-2002 is too big (glaciers unacounted for)
inlier_mask_2nd_iteration = ~glacier_outlines.create_mask(dem)
# Combine the inlier mask and slope mask
inlier_mask_2nd_iteration.data = inlier_mask_2nd_iteration.data & slope_mask & (dem.data < 1000) & (dem.data > 1)
# Remove invalid entries from the inlier mask
inlier_mask_2nd_iteration.data[d2012.data.mask==True]=False

pipeline = xdem.coreg.VerticalShift() + xdem.coreg.NuthKaab() + xdem.coreg.Deramp(poly_order=2)
d1999_coregistered = pipeline.fit_and_apply(reference_elev=d2002_coregistered, to_be_aligned_elev=d1999_coregistered, inlier_mask=inlier_mask_2nd_iteration, random_state=42)  # Coregister to d2002 because somehow icesat fails

In [ ]:
# Save the coregistered DEMs
d2012_coregistered.save('../Data_Paper/DEMs/d2012_coregistered.tif')
d1999_coregistered.save('../Data_Paper/DEMs/d1999_coregistered.tif')
d2002_coregistered.save('../Data_Paper/DEMs/d2002_coregistered.tif')

## 5) ICESat-2-derived DEMs (2019, 2022)

2019 and 2022 points adjusted to 30 August assuming linear change since the 2012 DEM (30 August 2012); dh/dt standardized by median and NMAD in 50 m bins of 2012 elevation, values beyond 3 standard deviations discarded; multiquadric RBF (epsilon = 500, smoothing = 100, 5 % random subsample), standardization reversed and added to the 2012 DEM. Writes `../Data_Paper/DEMs/d2019_icesat2.tif`, `d2022_icesat2.tif`.

In [ ]:
d2012_coregistered = xdem.DEM('../Data_Paper/DEMs/d2012_coregistered.tif')
gdf_no_outliers = gpd.read_file('../Data_Paper/ICESat-2/icesat_2_data_no_outliers.shp')
gdf_combined = gdf_no_outliers.copy()

d2012_coregistered.data.data[d2012_coregistered.data.data<0] = np.nan 

In [ ]:
def linear_dhdt(z_pt, z_ref, time_pt, time_ref, time_guess):

    interp_z = np.full(len(z_pt), np.nan)  # Initialize interpolated height array

    for i in tqdm(range(len(z_pt)), desc="Calculating linear dh/dt"):
        delta_t = (time_pt[i] - time_ref).days / 365.25 # Convert time difference to years
        delta_z = z_pt[i] - z_ref[i]  # Calculate the change in height

        a = delta_z/(delta_t)  # Calculate the slope of the linear change in height

        x_guess = (time_guess - time_ref).days / 365.25
        interp_z[i] = (a*x_guess)+z_ref[i]  # Calculate the dh/dt at the guess time
    
    # Calculate the linear change in dh/dt
    # dhdt_ref is 0, so intercept is zero. Only the slope matters, which is dhdt_pt

    return interp_z


gdf_2019 = gdf_combined[(gdf_combined['datetime'] >= '2019-01-01') & (gdf_combined['datetime'] < '2020-01-01')]
gdf_2022 = gdf_combined[(gdf_combined['datetime'] >= '2022-01-01') & (gdf_combined['datetime'] < '2023-01-01')]



comp_19 = d2012_coregistered.interp_points(points=(gdf_2019['geometry'].x, gdf_2019['geometry'].y))
time_ref = pd.to_datetime('2012-08-30')
time_guess = pd.to_datetime('2019-08-30')

gdf_2019['h_li_interpolated'] = linear_dhdt(gdf_2019['h_li'].values, comp_19, gdf_2019['datetime'].values, time_ref, time_guess)

comp_22 = d2012_coregistered.interp_points(points=(gdf_2022['geometry'].x, gdf_2022['geometry'].y))
time_ref = pd.to_datetime('2012-08-30')
time_guess = pd.to_datetime('2022-08-30')
gdf_2022['h_li_interpolated'] = linear_dhdt(gdf_2022['h_li'].values, comp_22, gdf_2022['datetime'].values, time_ref, time_guess)

In [ ]:
# Calculate NMAD function
def calculate_nmad(values):
    """NMAD = 1.4826 * median(|values - median(values)|)"""
    median_val = np.median(values)
    mad = np.median(np.abs(values - median_val))
    return 1.4826 * mad


def normalize_dhdt(pts_gdf, vals_interp, var):


    # Create bins of 100m interval between 0m and 00m
    bins = np.arange(0, 5000, 50)
    bins_centers = (bins[:-1] + bins[1:]) / 2

    # Assign each point to elevation bins
    bin_indices = np.digitize(vals_interp, bins) - 1

    # Filter valid bin indices
    valid_mask = (bin_indices >= 0) & (bin_indices < len(bins) - 1)
    valid_bin_indices = bin_indices[valid_mask]
    valid_dhdt = pts_gdf[var][valid_mask]
    valid_elevations = vals_interp[valid_mask]

    # Calculate NMAD and median for each bin
    bin_nmads = []
    bin_medians = []
    bin_centers_with_data = []

    for i in range(len(bins) - 1):
        bin_mask = valid_bin_indices == i
        bin_dhdt = valid_dhdt[bin_mask]
        
        if len(bin_dhdt) > 1:  # Need at least 2 points for NMAD
            # Remove NaN values
            clean_dhdt = bin_dhdt[~np.isnan(bin_dhdt)]
            if len(clean_dhdt) > 1:
                nmad = calculate_nmad(clean_dhdt)
                bin_nmads.append(nmad)
                bin_medians.append(np.median(clean_dhdt))
                bin_centers_with_data.append(bins_centers[i])

    # Convert to arrays for interpolation
    bin_nmads = np.array(bin_nmads)
    bin_medians = np.array(bin_medians)
    bin_centers_with_data = np.array(bin_centers_with_data)

    print(f"Calculated NMAD and median for {len(bin_nmads)} bins")
    print(f"Elevation range with data: {valid_elevations.min():.1f} - {valid_elevations.max():.1f} m")

    # Create linear interpolation function for NMAD
    

    if len(bin_centers_with_data) >= 2:
        # Linear interpolation with extrapolation
        nmad_interp = interp1d(bin_centers_with_data, bin_nmads, 
                            kind='linear', 
                            bounds_error=False, 
                            fill_value='extrapolate')
        
        median_interp = interp1d(bin_centers_with_data, bin_medians,
                            kind='linear', 
                            bounds_error=False, 
                            fill_value='extrapolate')
        
        # Get interpolated NMAD for each point's elevation
        interpolated_nmads = nmad_interp(valid_elevations)
        interpolated_medians = median_interp(valid_elevations)
        
        # Normalize dhdt by interpolated NMAD
        dhdt_normalized = (valid_dhdt - interpolated_medians)/ interpolated_nmads
        
        print(f"Normalized {len(dhdt_normalized)} dhdt values")
        print(f"Original dhdt range: {np.nanmin(valid_dhdt):.6f} to {np.nanmax(valid_dhdt):.6f}")
        print(f"Normalized dhdt range: {np.nanmin(dhdt_normalized):.6f} to {np.nanmax(dhdt_normalized):.6f}")
        print(f"NMAD range: {bin_nmads.min():.6f} to {bin_nmads.max():.6f}")
        
    else:
        print("Error: Not enough bins with data for interpolation")
        dhdt_normalized = None

    # Optional: Create a summary DataFrame of bin statistics
    bin_summary = []
    for i in range(len(bins) - 1):
        bin_mask = valid_bin_indices == i
        bin_dhdt = valid_dhdt[bin_mask]
        
        if len(bin_dhdt) > 0:
            clean_dhdt = bin_dhdt[~np.isnan(bin_dhdt)]
            if len(clean_dhdt) > 1:
                nmad = calculate_nmad(clean_dhdt)
                bin_summary.append({
                    'bin_center': bins_centers[i],
                    'bin_min': bins[i],
                    'bin_max': bins[i + 1],
                    'count': len(clean_dhdt),
                    'mean_dhdt': np.mean(clean_dhdt),
                    'median_dhdt': np.median(clean_dhdt),
                    'nmad': nmad
                })

    bin_summary_df = pd.DataFrame(bin_summary)
    print("\nBin summary (first 10 rows):")
    print(bin_summary_df.head(10))

    pts_gdf[f'{var}_standardized'] = dhdt_normalized

    return pts_gdf, nmad_interp, median_interp # Return updated gdf and interpolation function

In [ ]:
def interpolate_icesat_raster(pts_gdf, dem_ref, factor, gridsize, sdate, edate, sm, eps, n_points_neighbor):

    
    # Interpolate reference DEM elevations at point locations
    vals_interp = dem_ref.interp_points(points=(pts_gdf["X"], pts_gdf["Y"]))

    print(f"[{datetime.now()}] Computing dh/dt...")
    # Compute rate of elevation change
    timedelta_diff = (edate - sdate).days / 365.25
    pts_gdf["dhdt"] = (pts_gdf["h_li_interpolated"] - vals_interp) / timedelta_diff

    print(f"[{datetime.now()}] Normalizing dh/dt by NMAD in elevation bins...")
    # Normalize dhdt using NMAD in elevation bins
    pts_gdf, nmad_func, median_func = normalize_dhdt(pts_gdf, vals_interp, 'dhdt')

    # Trim the outliers by keeping only values within 3 standard deviations of the mean and setting them to NaN
    mean_dhdt = np.nanmean(pts_gdf['dhdt_standardized'])
    std_dhdt = np.nanstd(pts_gdf['dhdt_standardized'])
    lower_bound = mean_dhdt - 3 * std_dhdt
    upper_bound = mean_dhdt + 3 * std_dhdt
    pts_gdf.loc[(pts_gdf['dhdt_standardized'] < lower_bound) | (pts_gdf['dhdt_standardized'] > upper_bound), 'dhdt_standardized'] = np.nan

    var = 'dhdt_standardized'

    rast = interpolate_icesat2(var, pts_gdf, dem_ref, factor, gridsize, 'multiquadric', epsilon = eps, sm=sm)

    rast = ((((rast.data.data * nmad_func(dem_ref.data.data)) + median_func(dem_ref.data.data)) * timedelta_diff) + dem_ref.reproject(rast))
    
    rast.set_vcrs("Ellipsoid")


    return rast

In [ ]:
factor = 20
gridsize = int(8000/3)
n_points_neighbor = 200
rast_2019 = interpolate_icesat_raster(gdf_2019, d2012_coregistered, factor, gridsize, pd.to_datetime('2012-08-30'), pd.to_datetime('2019-08-30'), 100, 500, n_points_neighbor)
rast_2022 = interpolate_icesat_raster(gdf_2022, d2012_coregistered, factor, gridsize, pd.to_datetime('2012-08-30'), pd.to_datetime('2022-08-30'), 100, 500, n_points_neighbor)

In [ ]:
rast_2019.save('../Data_Paper/DEMs/d2019_icesat2.tif')
rast_2022.save('../Data_Paper/DEMs/d2022_icesat2.tif')